In [3]:
# Carrega o motor08 direto do repositorio oficial do professor
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor08.R")
docs <- docs_aula()          # os 8 documentos
cfg  <- cfg_aula()           # decisoes canonicas (janela 3)
ix   <- montar(docs, cfg)    # o indice

In [4]:
rb <- ranking_bm25("modelo de recuperacao", ix, cfg)        # estagio 1
qt <- preparar("modelo de recuperacao", cfg)                # modelo, de, recuperacao
px <- sapply(ix$tokens, function(dt) proximidade(qt, dt))   # estagio 2: d2 1, d3 0,5[cite: 1]
g  <- gabarito_aula()                                       # os graus[cite: 1]
curva <- t(sapply(1:8, function(k) metricas(reordenar(rb, k, px), g, 2)))   # uma linha por k[cite: 1]
rownames(curva) <- paste0("k", 1:8)                         # nomeia as linhas[cite: 1]
round(curva[, c("AP", "nDCG_grad")], 3)                     # qualidade em cada k[cite: 1]

ERROR: Error in proximidade(qt, dt): could not find function "proximidade"


In [5]:
# Funcoes da Aula 08 (caso nao tenha copiado do seu notebook aula08.ipynb)
proximidade <- function(qt, dt, janela = 3) {
  q <- unique(intersect(qt, dt))
  if (length(q) < 2) return(0)
  pos <- lapply(q, function(t) which(dt == t))
  pares <- 0
  prox  <- 0
  for (i in 1:(length(q) - 1)) {
    for (j in (i + 1):length(q)) {
      pares <- pares + 1
      dmin <- min(abs(outer(pos[[i]], pos[[j]], "-")))
      if (dmin <= janela) prox <- prox + 1 / dmin
    }
  }
  prox / pares
}

reordenar <- function(rk, k, sc) {
  if (k <= 1) return(names(rk))
  cab <- names(rk)[1:k]
  ord <- cab[order(sc[cab], decreasing = TRUE)]
  c(ord, names(rk)[-(1:k)])
}

In [6]:
rb <- ranking_bm25("modelo de recuperacao", ix, cfg)        # estagio 1
qt <- preparar("modelo de recuperacao", cfg)                # modelo, de, recuperacao
px <- sapply(ix$tokens, function(dt) proximidade(qt, dt))   # estagio 2: d2 1, d3 0,5
g  <- gabarito_aula()                                       # os graus[cite: 1]
curva <- t(sapply(1:8, function(k) metricas(reordenar(rb, k, px), g, 2)))   # uma linha por k[cite: 1]
rownames(curva) <- paste0("k", 1:8)                         # nomeia as linhas[cite: 1]
round(curva[, c("AP", "nDCG_grad")], 3)                     # qualidade em cada k[cite: 1]


,AP,nDCG_grad
k1,0.833,0.951
k2,0.583,0.863
k3,0.583,0.863
k4,0.583,0.863
k5,0.583,0.863
k6,0.583,0.863
k7,0.583,0.863
k8,0.583,0.863


In [7]:
# Restaura as funcoes oficiais do motor08.R
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor08.R")

In [8]:
rb <- ranking_bm25("modelo de recuperacao", ix, cfg)        # estagio 1
qt <- preparar("modelo de recuperacao", cfg)                # modelo, de, recuperacao
px <- sapply(ix$tokens, function(dt) proximidade(qt, dt))   # estagio 2: d2 1, d3 0,5
g  <- gabarito_aula()                                       # os graus[cite: 1]
curva <- t(sapply(1:8, function(k) metricas(reordenar(rb, k, px), g, 2)))   # uma linha por k[cite: 1]
rownames(curva) <- paste0("k", 1:8)                         # nomeia as linhas[cite: 1]
round(curva[, c("AP", "nDCG_grad")], 3)                     # qualidade em cada k[cite: 1]

,AP,nDCG_grad
k1,0.833,0.951
k2,0.583,0.863
k3,0.583,0.863
k4,0.583,0.863
k5,0.583,0.863
k6,0.583,0.863
k7,0.583,0.863
k8,0.583,0.863


In [9]:
# Garante que reordenar aceite tanto vetor nomeado completo quanto vetor de tamanho k
reordenar <- function(rk, k, sc) {
  if (k <= 1) return(names(rk))
  cab <- names(rk)[1:k]
  s   <- if (!is.null(names(sc)) && all(cab %in% names(sc))) sc[cab] else sc[1:k]
  ord <- cab[order(s, decreasing = TRUE)]
  c(ord, names(rk)[-(1:k)])
}

In [10]:
rerank_prox <- function(rk, qt, tokens, k, janela = 3) {   # estagio 2 honesto[cite: 1]
  if (k == 0) return(names(rk))                            # k = 0: nao reordena[cite: 1]
  top <- head(names(rk), k)                                # os candidatos[cite: 1]
  sc  <- sapply(tokens[top], function(dt) proximidade(qt, dt, janela))   # so eles sao lidos[cite: 1]
  reordenar(rk, k, sc)                                     # cabeca + cauda[cite: 1]
}

tempo <- sapply(0:8, function(k)                                   # para cada k...[cite: 1]
  system.time(for (i in 1:5000) rerank_prox(rb, qt, ix$tokens, k))[["elapsed"]])   # ...5000 vezes[cite: 1]
names(tempo) <- paste0("k", 0:8)                                   # nomeia[cite: 1]
tempo                                                              # segundos por 5000 consultas[cite: 1]

k0    k1    k2    k3    k4    k5    k6    k7    k8 
0.014 0.498 1.121 1.423 2.193 2.652 2.247 2.327 2.567

In [11]:
r <- relevancia(rb, g, 2)                                     # relevantes na ordem do BM25[cite: 1]
sapply(1:8, function(k) recall_k(r$rel, k, r$R))              # R@k: o teto[cite: 1]
rc <- ranking_cosseno("modelo de recuperacao", ix, cfg)       # outro estagio 1 (Aula 02)[cite: 1]
names(rc)                                                     # onde ficou o d2?[cite: 1]
round(sapply(1:8, function(k) metricas(reordenar(rc, k, px), g, 2)[["AP"]]), 3)   # a curva[cite: 1]

[1] 0.5 0.5 1.0 1.0 1.0 1.0 1.0 1.0

[1] "d1" "d3" "d4" "d2" "d6" "d8" "d5" "d7"

[1] 0.500 0.500 0.500 0.583 0.583 0.583 0.583 0.583

In [12]:
if (is.null(cfg$modelo)) cfg$modelo <- "bm25"   # estagio 1 = BM25
if (is.null(cfg$lsa_k))  cfg$lsa_k  <- 2        # so usado com modelo "lsa" ou "hibrido"
if (is.null(cfg$rrf_c))  cfg$rrf_c  <- 60       # so usado com modelo "hibrido"
if (is.null(cfg$janela)) cfg$janela <- 3        # janela da proximidade (Aula 08)

estagio1 <- function(texto, ix, cfg, m) {                       # o motor do grupo (Parte D)
  rb <- ranking_bm25(texto, ix, cfg)                            # BM25 com k1 e b do grupo
  if (cfg$modelo == "bm25") return(rb)                          # "bm25": pronto
  rl <- ranking_lsa(texto, ix, cfg, m)                          # LSA da Aula 07
  if (cfg$modelo == "lsa") rl else rrf(rb, rl, c = cfg$rrf_c)   # "lsa" ou hibrido
}

m <- if (cfg$modelo == "bm25") NULL else lsa(ix, cfg$lsa_k)     # LSA so se usado
dev   <- intersect(usar, nec$consulta[nec$conjunto == "dev"])          # aqui se escolhe
teste <- intersect(usar, nec$consulta[nec$conjunto == "teste"])        # so no fim
rks <- lapply(consultas[usar], function(tx) estagio1(tx, ix, cfg, m))  # estagio 1, uma vez
qts <- lapply(consultas[usar], function(tx) preparar(tx, cfg))         # tokens do grupo
seg1 <- system.time(for (q in dev) estagio1(consultas[[q]], ix, cfg, m))[["elapsed"]]
1000 * seg1 / length(dev)                                              # T1: ms por consulta

ERROR: Error: object 'usar' not found


In [13]:
# Carrega o corpus e as configuracoes do grupo Predicarga diretamente do GitHub
REPO <- "https://raw.githubusercontent.com/andrecesar00/predicarga_andre_diego_kaykhy/main"
source(paste0(REPO, "/estrutura/codigo/config.R"))

cp        <- read.csv(paste0(REPO, "/estrutura/dados/corpus.csv"), stringsAsFactors = FALSE)
docs      <- setNames(cp$texto, cp$id)
ix        <- montar(docs, cfg)

nec       <- read.csv(paste0(REPO, "/estrutura/dados/necessidades.csv"), stringsAsFactors = FALSE)
consultas <- setNames(nec$texto_consulta, nec$consulta)
qrels     <- ler_qrels(paste0(REPO, "/estrutura/dados/qrels_Andre_2026-10-25.csv"))
usar      <- intersect(names(consultas), names(qrels))

Warning message in file(file, "rt"):
“cannot open URL 'https://raw.githubusercontent.com/andrecesar00/predicarga_andre_diego_kaykhy/main/estrutura/dados/corpus.csv': HTTP status was '404 Not Found'”


ERROR: Error in file(file, "rt"): cannot open the connection to 'https://raw.githubusercontent.com/andrecesar00/predicarga_andre_diego_kaykhy/main/estrutura/dados/corpus.csv'


In [14]:
# 1. Carrega motor, configuracoes e corpus do grupo Predicarga
REPO <- "https://raw.githubusercontent.com/andrecesar00/predicarga_andre_diego_kaykhy/main/"
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor08.R")
download.file(paste0(REPO, "estrutura/codigo/config.R"), "config.R", quiet = TRUE)
source("config.R")

cp     <- read.csv(paste0(REPO, "estrutura/banco-de-dados/corpus.csv"), stringsAsFactors = FALSE, fileEncoding = "UTF-8")
docs   <- setNames(cp$texto, cp$id)
ix     <- montar(docs, cfg)

arqs <- c("qrels_Andre_2026-10-25.csv")
for (a in arqs) download.file(paste0(REPO, "estrutura/banco-de-dados/", a), a, quiet = TRUE)
qrels <- ler_qrels(arqs, juiz = cfg$juizes)
nec   <- read.csv(paste0(REPO, "estrutura/banco-de-dados/necessidades.csv"), stringsAsFactors = FALSE, fileEncoding = "UTF-8-BOM")
consultas <- setNames(nec$texto_consulta, nec$consulta)
R_q  <- sapply(qrels, function(g) sum(g >= cfg$limiar))
usar <- names(qrels)[R_q > 0]

# 2. Restaura funcoes exatas da Aula 08 do grupo
proximidade <- function(qt, dt, janela = 3) {
  s <- 0
  for (i in seq_along(qt)) for (j in seq_along(qt))
    if (i < j && qt[i] != qt[j]) {
      pos_i <- which(dt == qt[i])
      pos_j <- which(dt == qt[j])
      if (length(pos_i) && length(pos_j)) {
        d <- outer(pos_j, pos_i, "-")
        d <- d[d > 0 & d <= janela]
        if (length(d)) s <- s + 1 / min(d)
      }
    }
  s
}

reordenar <- function(rk, k, sc) {
  k <- min(k, length(rk))
  top <- names(rk)[1:k]
  nova <- top[order(-sc[top], seq_along(top))]
  c(nova, names(rk)[-(1:k)])
}

estagio1 <- function(texto, ix, cfg, m) {
  rb <- ranking_bm25(texto, ix, cfg)
  if (cfg$modelo == "bm25") return(rb)
  rl <- ranking_lsa(texto, ix, cfg, m)
  if (cfg$modelo == "lsa") rl else rrf(rb, rl, c = cfg$rrf_c)
}
m <- if (cfg$modelo == "bm25") NULL else lsa(ix, cfg$lsa_k)

rerank_prox <- function(rk, qt, tokens, k, janela = 3) {
  if (k == 0) return(names(rk))
  top <- head(names(rk), k)
  sc  <- sapply(tokens[top], function(dt) proximidade(qt, dt, janela))
  reordenar(rk, k, sc)
}

# 3. Medicao no dev: T1 e tabela k x MAP x nDCG x ms
dev   <- intersect(usar, nec$consulta[nec$conjunto == "dev"])
teste <- intersect(usar, nec$consulta[nec$conjunto == "teste"])
rks <- lapply(consultas[usar], function(tx) estagio1(tx, ix, cfg, m))
qts <- lapply(consultas[usar], function(tx) preparar(tx, cfg))

seg1 <- system.time(for (i in 1:200) for (q in dev) estagio1(consultas[[q]], ix, cfg, m))[["elapsed"]]
cat("T1 (ms por consulta no dev):", round(1000 * seg1 / (200 * length(dev)), 3), "\n\n")

med <- function(q, k, rks, qts, qrels, tokens, w, limiar)
  metricas(rerank_prox(rks[[q]], qts[[q]], tokens, k, w), qrels[[q]], limiar)[c("AP", "nDCG_grad")]

w  <- 3
ks <- c(0, 1, 2, 3, 5, 10, 20, 50, 100)
ks <- ks[ks <= ix$N]

tab <- t(sapply(ks, function(k) {
  seg <- system.time(for (i in 1:200) for (q in dev)
    rerank_prox(rks[[q]], qts[[q]], ix$tokens, k, w))[["elapsed"]]
  mt <- rowMeans(sapply(dev, function(q) med(q, k, rks, qts, qrels, ix$tokens, w, cfg$limiar)))
  c(k = k, MAP = mt[["AP"]], nDCG = mt[["nDCG_grad"]], ms = 1000 * seg / (200 * length(dev)))
}))
round(tab, 3)

T1 (ms por consulta no dev): 0.862 



k,MAP,nDCG,ms
0,0.066,0.268,0.002
1,0.066,0.268,0.175
2,0.066,0.268,0.178
3,0.066,0.268,0.243
5,0.066,0.268,0.343
10,0.066,0.268,0.578
20,0.066,0.268,1.025
50,0.066,0.268,2.133


In [15]:
# Verifica se alguma consulta no dev mudou o AP ou a ordem do top-5 com k = 50
ap0 <- sapply(dev, function(q) med(q, 0, rks, qts, qrels, ix$tokens, w, cfg$limiar)[["AP"]])
ap1 <- sapply(dev, function(q) med(q, 50, rks, qts, qrels, ix$tokens, w, cfg$limiar)[["AP"]])
cat("Ajudou em:", dev[ap1 > ap0], "| Atrapalhou em:", dev[ap1 < ap0], "\n\n")

# Compara o top-5 de q01 e q03 entre k = 0 e k = 50 (e le o primeiro texto)
for (q in c("q01", "q03")) {
  cat("Consulta", q, ":\n")
  print(rbind(antes  = names(rks[[q]])[1:5],
              depois = rerank_prox(rks[[q]], qts[[q]], ix$tokens, 50, w)[1:5]))
}
cat("\nTexto do 1o colocado em q01:", substr(docs[names(rks[["q01"]])[1]], 1, 180), "...\n")

Ajudou em:  | Atrapalhou em:  

Consulta q01 :
       [,1] [,2] [,3] [,4] [,5]
antes  "d6" "d2" "d4" "d1" "d3"
depois "d6" "d2" "d4" "d1" "d3"
Consulta q03 :
       [,1]  [,2] [,3]  [,4]  [,5] 
antes  "d53" "d4" "d64" "d57" "d62"
depois "d53" "d4" "d64" "d57" "d62"

Texto do 1o colocado em q01: Nas primeiras décadas do século XVI, existiu, onde hoje é o centro de São Vicente, um povoado, fundado pelo enigmático Bacharel de Cananeia, um degredado português que se fixou no  ...


In [16]:
k_esc <- 0                                                     # escolha do grupo no dev
round(sapply(teste, function(q) c(antes  = med(q, 0, rks, qts, qrels, ix$tokens, w, cfg$limiar),
                                  depois = med(q, k_esc, rks, qts, qrels, ix$tokens, w, cfg$limiar))), 3)

,q04,q05
antes.AP,0.112,0.071
antes.nDCG_grad,0.362,0.322
depois.AP,0.112,0.071
depois.nDCG_grad,0.362,0.322


In [17]:
A matrix: 4 × 2 of type dbl
q04	q05
antes.AP	0.112	0.071
antes.nDCG_grad	0.362	0.322
depois.AP	0.112	0.071
depois.nDCG_grad	0.362	0.322


ERROR: Error in parse(text = input): <text>:1:3: unexpected symbol
1: A matrix
      ^


In [18]:
k_esc <- 0
round(sapply(teste, function(q) c(antes  = med(q, 0, rks, qts, qrels, ix$tokens, w, cfg$limiar),
                                  depois = med(q, k_esc, rks, qts, qrels, ix$tokens, w, cfg$limiar))), 3)

,q04,q05
antes.AP,0.112,0.071
antes.nDCG_grad,0.362,0.322
depois.AP,0.112,0.071
depois.nDCG_grad,0.362,0.322
